# 📧 Spam Email Classifier
### Fundamentals of AI and ML — BYOP Capstone Project

**Objective:** Build a machine learning model that classifies SMS/email messages as **Spam** or **Ham (Not Spam)** using Natural Language Processing and Naive Bayes classification.

**Dataset:** SMS Spam Collection Dataset (UCI Machine Learning Repository)  
**Techniques Used:** Text Preprocessing, TF-IDF Vectorization, Naive Bayes, Logistic Regression, Model Evaluation

## Step 1: Install & Import Libraries

In [ ]:
# Install required libraries
!pip install nltk scikit-learn pandas matplotlib seaborn wordcloud -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

nltk.download('stopwords')
nltk.download('punkt')

print('✅ All libraries imported successfully!')

## Step 2: Load the Dataset

In [ ]:
# Load dataset directly from UCI repository
url = 'https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv'
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

print(f'Dataset shape: {df.shape}')
print(f'\nFirst 5 rows:')
df.head()

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution
print('Label Distribution:')
print(df['label'].value_counts())
print(f'\nSpam %: {df["label"].value_counts(normalize=True)["spam"]*100:.1f}%')
print(f'Ham  %: {df["label"].value_counts(normalize=True)["ham"]*100:.1f}%')

# Plot class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'salmon'], edgecolor='black')
axes[0].set_title('Message Count by Class', fontsize=13)
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
axes[0].tick_params(rotation=0)

# Pie chart
df['label'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                 colors=['steelblue', 'salmon'], startangle=90)
axes[1].set_title('Proportion of Spam vs Ham', fontsize=13)
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot saved!')

In [ ]:
# Message length analysis
df['msg_length'] = df['message'].apply(len)

fig, ax = plt.subplots(figsize=(10, 4))
df[df['label']=='ham']['msg_length'].plot(kind='hist', bins=50, alpha=0.7,
                                           color='steelblue', label='Ham', ax=ax)
df[df['label']=='spam']['msg_length'].plot(kind='hist', bins=50, alpha=0.7,
                                            color='salmon', label='Spam', ax=ax)
ax.set_title('Message Length Distribution', fontsize=13)
ax.set_xlabel('Message Length (characters)')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.savefig('length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Average Ham length:  {df[df['label']=='ham']['msg_length'].mean():.0f} chars")
print(f"Average Spam length: {df[df['label']=='spam']['msg_length'].mean():.0f} chars")

In [ ]:
# Word Cloud for Spam messages
spam_text = ' '.join(df[df['label'] == 'spam']['message'])
ham_text  = ' '.join(df[df['label'] == 'ham']['message'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

wc_spam = WordCloud(width=600, height=400, background_color='white',
                    colormap='Reds', max_words=100).generate(spam_text)
axes[0].imshow(wc_spam, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('Most Common Words in SPAM', fontsize=13, color='darkred')

wc_ham = WordCloud(width=600, height=400, background_color='white',
                   colormap='Blues', max_words=100).generate(ham_text)
axes[1].imshow(wc_ham, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Most Common Words in HAM', fontsize=13, color='steelblue')

plt.tight_layout()
plt.savefig('wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4: Text Preprocessing

In [ ]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """Clean and normalize text for ML processing."""
    # 1. Lowercase
    text = text.lower()
    # 2. Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # 3. Remove special characters and digits
    text = re.sub(r'[^a-z\s]', '', text)
    # 4. Tokenize
    tokens = text.split()
    # 5. Remove stopwords and apply stemming
    tokens = [stemmer.stem(word) for word in tokens if word not in stop_words]
    return ' '.join(tokens)

# Apply preprocessing
df['clean_message'] = df['message'].apply(preprocess_text)

# Show before/after example
print('=== Preprocessing Example ===')
sample = df[df['label']=='spam'].iloc[0]
print(f'Original : {sample["message"]}')
print(f'Processed: {sample["clean_message"]}')

## Step 5: Feature Extraction (TF-IDF)

In [ ]:
# Encode labels
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_message'], df['label_num'],
    test_size=0.2, random_state=42, stratify=df['label_num']
)

print(f'Training samples : {len(X_train)}')
print(f'Testing  samples : {len(X_test)}')

# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'\nTF-IDF feature matrix shape: {X_train_tfidf.shape}')
print('✅ Features extracted!')

## Step 6: Train Models

In [ ]:
# Model 1: Multinomial Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_preds = nb_model.predict(X_test_tfidf)

print('=== Naive Bayes Results ===')
print(f'Accuracy: {accuracy_score(y_test, nb_preds)*100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, nb_preds, target_names=['Ham', 'Spam']))

In [ ]:
# Model 2: Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)
lr_preds = lr_model.predict(X_test_tfidf)

print('=== Logistic Regression Results ===')
print(f'Accuracy: {accuracy_score(y_test, lr_preds)*100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, lr_preds, target_names=['Ham', 'Spam']))

## Step 7: Model Evaluation & Visualizations

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, preds, title in zip(axes,
                             [nb_preds, lr_preds],
                             ['Naive Bayes', 'Logistic Regression']):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Ham', 'Spam'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'Confusion Matrix — {title}', fontsize=12)

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Model Comparison Bar Chart
from sklearn.metrics import precision_score, recall_score, f1_score

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
nb_scores = [
    accuracy_score(y_test, nb_preds),
    precision_score(y_test, nb_preds),
    recall_score(y_test, nb_preds),
    f1_score(y_test, nb_preds)
]
lr_scores = [
    accuracy_score(y_test, lr_preds),
    precision_score(y_test, lr_preds),
    recall_score(y_test, lr_preds),
    f1_score(y_test, lr_preds)
]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, nb_scores, width, label='Naive Bayes', color='steelblue')
bars2 = ax.bar(x + width/2, lr_scores, width, label='Logistic Regression', color='salmon')

ax.set_ylim(0.85, 1.02)
ax.set_ylabel('Score')
ax.set_title('Model Comparison: Naive Bayes vs Logistic Regression', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()

for bar in bars1 + bars2:
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8: Top Features (What Makes a Message Spam?)

In [ ]:
# Top spam-indicating words from Naive Bayes
feature_names = tfidf.get_feature_names_out()
log_probs = nb_model.feature_log_prob_

spam_top = np.argsort(log_probs[1])[-20:][::-1]
ham_top  = np.argsort(log_probs[0])[-20:][::-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh([feature_names[i] for i in spam_top[::-1]],
             [log_probs[1][i] for i in spam_top[::-1]], color='salmon')
axes[0].set_title('Top 20 Spam Indicator Words', fontsize=12)
axes[0].set_xlabel('Log Probability')

axes[1].barh([feature_names[i] for i in ham_top[::-1]],
             [log_probs[0][i] for i in ham_top[::-1]], color='steelblue')
axes[1].set_title('Top 20 Ham Indicator Words', fontsize=12)
axes[1].set_xlabel('Log Probability')

plt.tight_layout()
plt.savefig('top_features.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9: Real-Time Spam Predictor

In [ ]:
def predict_spam(message, model=nb_model, vectorizer=tfidf):
    """Predict whether a message is spam or ham."""
    cleaned = preprocess_text(message)
    vectorized = vectorizer.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    probability = model.predict_proba(vectorized)[0]

    label = '🚨 SPAM' if prediction == 1 else '✅ HAM (Not Spam)'
    confidence = probability[prediction] * 100

    print(f'Message   : "{message}"')
    print(f'Prediction: {label}')
    print(f'Confidence: {confidence:.1f}%')
    print('-' * 60)

# Test with sample messages
test_messages = [
    "Congratulations! You've won a FREE iPhone. Click here to claim now!",
    "Hey, are we still meeting at 5pm today?",
    "URGENT: Your account will be suspended. Verify now at bit.ly/xyz",
    "Can you pick up some groceries on your way home?",
    "Win £1000 cash! Text WIN to 87121 now. Free entry, 18+"
]

print('=== Spam Predictor Demo ===')
print()
for msg in test_messages:
    predict_spam(msg)

In [ ]:
# Try your own message!
user_message = input('Enter a message to test: ')
predict_spam(user_message)

## Step 10: Summary

| Model | Accuracy | Notes |
|-------|----------|-------|
| Multinomial Naive Bayes | ~98% | Fast, interpretable, great for text |
| Logistic Regression | ~98% | Slightly better precision on some classes |

**Key Takeaways:**
- Spam messages are significantly longer than ham messages on average
- Words like "free", "win", "prize", "claim", "urgent" are strong spam indicators
- TF-IDF with bigrams captures both individual words and common spam phrases
- Naive Bayes is a strong baseline for text classification problems
- Both models achieve ~98% accuracy, showing ML is highly effective for spam detection